# 04 — RL Agent Training & Evaluation

> **Portfolio Optimizer** · *Reinforcement Learning Portfolio Management*

**Objectives**
1. Build the `NSEPortfolioEnv` Gymnasium environment — discrete action space (buy/sell/hold × 12 tickers).
2. Train a PPO agent with 500,000 timesteps on 3 years of NSE data.
3. Evaluate on a 1-year out-of-sample holdout period.
4. Compare cumulative returns, Sharpe, and max drawdown vs NIFTY50 and equal-weight benchmarks.
5. Demonstrate ≥15% alpha over NIFTY50 on the holdout.
6. Visualise agent decision heatmaps and portfolio evolution.

In [ ]:
import warnings, os, sys
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import stable_baselines3 as sb3

NAVY, SLATE, ORANGE, GREY = '#1A273A', '#3E4A62', '#C24D2C', '#D9D9D7'
def base_layout(**kw):
    return dict(paper_bgcolor=NAVY, plot_bgcolor=NAVY,
                font=dict(color=GREY), xaxis=dict(gridcolor=SLATE),
                yaxis=dict(gridcolor=SLATE), **kw)

TICKERS = [
    'RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS',
    'ICICIBANK.NS','WIPRO.NS','BAJFINANCE.NS','ASIANPAINT.NS',
    'TITAN.NS','MARUTI.NS','ONGC.NS','ZOMATO.NS',
]
TRAIN_END = '2022-12-31'
TEST_START = '2023-01-01'
print(f'SB3 version: {sb3.__version__}  |  Tickers: {len(TICKERS)}')

In [ ]:
# ── Load data for all tickers ─────────────────────────────────────────────────
from data.data_ingestion import load_processed

all_data = {}
for ticker in TICKERS:
    try:
        all_data[ticker] = load_processed(ticker)
        print(f'  {ticker}: {all_data[ticker].shape[0]} rows')
    except Exception as e:
        print(f'  {ticker}: MISSING ({e})')

available = list(all_data.keys())
print(f'\nLoaded {len(available)}/{len(TICKERS)} tickers')

In [ ]:
# ── Initialise RL environment ─────────────────────────────────────────────────
from models.rl_agent import NSEPortfolioEnv

# Use available tickers, fall back to synthetic if needed
env = NSEPortfolioEnv(
    tickers=available[:4] if len(available) >= 4 else TICKERS[:4],
    mode='train',
    initial_capital=1_000_000,  # ₹10 Lakhs
    transaction_cost=0.001,
    reward_scaling=1e-4,
)

obs, info = env.reset()
print(f'Observation space: {env.observation_space}')
print(f'Action space:      {env.action_space}')
print(f'Obs shape:         {obs.shape}')

In [ ]:
# ── Train PPO agent ───────────────────────────────────────────────────────────
from models.rl_agent import train_ppo_agent

agent = train_ppo_agent(
    env=env,
    total_timesteps=100_000,   # reduced for demo; use 500_000 for production
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
    save_path='../rl_model/ppo_nse_portfolio',
    verbose=1,
)
print('\n✅ Agent training complete.')

In [ ]:
# ── Backtest on hold-out period ───────────────────────────────────────────────
from models.rl_agent import backtest_rl_agent

results = backtest_rl_agent(
    agent=agent,
    tickers=available[:4] if len(available) >= 4 else TICKERS[:4],
    start_date=TEST_START,
    end_date='2023-12-31',
    initial_capital=1_000_000,
)

print('\n── Backtest Results ─────────────────────────────────────────')
for k, v in results['summary'].items():
    print(f'  {k:30s}: {v}')

In [ ]:
# ── Cumulative returns comparison ─────────────────────────────────────────────
perf = results.get('equity_curve', pd.DataFrame())

if not perf.empty:
    fig = go.Figure()
    for strat, col in [('RL Agent', ORANGE), ('NIFTY50', GREY), ('Equal Weight', SLATE)]:
        if strat in perf.columns:
            fig.add_trace(go.Scatter(
                x=perf.index, y=perf[strat],
                name=strat, line=dict(color=col, width=2),
            ))

    fig.update_layout(**base_layout(
        title='RL Agent vs Benchmarks — Cumulative Return (%)', height=500,
        yaxis_title='Return (%)',
    ))
    fig.show()
else:
    print('Equity curve not available — check backtest_rl_agent output.')

In [ ]:
# ── Action distribution heatmap ───────────────────────────────────────────────
actions = results.get('actions', None)
if actions is not None and hasattr(actions, 'shape'):
    # actions shape: (T, n_tickers)  values in [-1=sell, 0=hold, +1=buy]
    avg_actions = pd.DataFrame(
        actions, columns=[t.replace('.NS','') for t in available[:actions.shape[1]]]
    ).mean()

    fig = go.Figure(go.Bar(
        x=avg_actions.index, y=avg_actions.values,
        marker_color=[ORANGE if v > 0 else SLATE for v in avg_actions.values],
    ))
    fig.update_layout(**base_layout(
        title='Mean Agent Action per Ticker (>0 = net-buy, <0 = net-sell)', height=400,
        yaxis_title='Mean Action',
    ))
    fig.show()
else:
    print('Action history not available from backtest.')

In [ ]:
# ── Performance summary table ─────────────────────────────────────────────────
summary = results.get('summary', {})
bench   = results.get('benchmarks', {})

rows = []
for name, d in [('RL Agent', summary), ('NIFTY50', bench.get('nifty50', {})),
                ('Equal Weight', bench.get('equal_weight', {}))]:
    if d:
        rows.append({
            'Strategy':         name,
            'Ann. Return (%)':  d.get('ann_return_pct', 'N/A'),
            'Sharpe':           d.get('sharpe',         'N/A'),
            'Max DD (%)':       d.get('max_drawdown_pct','N/A'),
            'Calmar':           d.get('calmar',          'N/A'),
        })

if rows:
    print(pd.DataFrame(rows).set_index('Strategy').to_string())
    alpha = (summary.get('ann_return_pct', 0) or 0) - (bench.get('nifty50', {}).get('ann_return_pct', 0) or 0)
    print(f'\n  Alpha vs NIFTY50: {alpha:+.1f}%')
else:
    print('Summary data unavailable — run full 500K timestep training for production results.')

print('\n✅  RL Agent notebook complete.')